# Task 1.1 — Core Contribution / Architecture
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov  

---
## Step-by-Step Method Description

The method proposed in this paper is a **gradient-based poisoning attack** against SVMs. The attacker crafts a single malicious training point that maximally degrades the SVM's classification accuracy on held-out validation data. Below is the full pipeline from input to output.

---
### Step 1: Define the Attacker's Objective

**What happens:**  
The attacker has access to the training set $D_{tr} = \{x_i, y_i\}_{i=1}^{n}$ and draws a validation set $D_{val} = \{x_k, y_k\}_{k=1}^{m}$. The goal is to find a single attack point $(x_c, y_c)$ such that, when injected into $D_{tr}$, it maximises the hinge loss on $D_{val}$:

$$\max_{x_c} \; L(x_c) = \sum_{k=1}^{m} (1 - y_k f_{x_c}(x_k))_+ = \sum_{k=1}^{m} (-g_k)_+$$

**Reference:** Equation (1), Section 2.1  

**Purpose:**  
Framing poisoning as a continuous optimisation problem (instead of a combinatorial search over all possible attack points) makes it tractable. The hinge loss is used as a surrogate for classification error because it is differentiable almost everywhere.

---
### Step 2: Initialise the Attack Point

**What happens:**  
A random point from the *attacked class* (the class the adversary wants to degrade) is selected and its label is flipped to the *attacking class*. This becomes the starting point $x_c^{(0)}$ for gradient ascent.

**Reference:** Algorithm 1 (initialisation step), Section 2.3  

**Purpose:**  
Starting inside the attacking class's margin ensures the initial point is already somewhat harmful. If the initialisation point is too close to the decision boundary, the attack point risks becoming a reserve point (with $\alpha_c = 0$), which would halt gradient computation. The label flip ensures the point is counted as a training error for the SVM, creating maximum disruption from the start.

---
### Step 3: Re-fit the SVM Using Incremental Learning

**What happens:**  
At each iteration $p$, the SVM is retrained on $D_{tr} \cup \{x_c^{(p)}, y_c\}$ using the **incremental SVM** of Cauwenberghs & Poggio (2001). This technique updates the dual variables $\alpha$ and bias $b$ efficiently without solving the full QP from scratch — it exploits the fact that adding one point causes a smooth change in the optimal solution.

**Reference:** Algorithm 1 line 4; adiabatic update condition, Equations (4)–(5), Section 2.1  

**Purpose:**  
The SVM solution must be exactly optimal at each step, because the gradient derivation in the next step relies on the KKT conditions holding. Using the incremental update keeps the solution on the optimal manifold while being computationally efficient.

---
### Step 4: Derive the Closed-Form Gradient via KKT Conditions

**What happens:**  
The core mathematical contribution of the paper is deriving a closed-form expression for $\partial L / \partial u$ — the gradient of the validation hinge loss with respect to the attack direction $u$. 

This is non-trivial because moving $x_c$ changes both:
1. The kernel matrix entry $Q_{kc}$ directly (since the kernel depends on $x_c$), and  
2. The SVM solution $(\alpha, b)$ indirectly, since the SVM must remain optimal.

The paper handles (2) by differentiating the KKT conditions (Eqs. 4–5) with respect to $u$, yielding the **adiabatic response**:

$$\begin{bmatrix} \partial b / \partial u_l \\ \partial \alpha_s / \partial u_l \end{bmatrix} = -\begin{bmatrix} 0 & y_S^\top \\ y_s & Q_{ss} \end{bmatrix}^{-1} \begin{bmatrix} 0 \\ \partial Q_{sc} / \partial u_l \end{bmatrix} \alpha_c$$

After substituting and simplifying using the Sherman–Morrison–Woodbury formula (Eq. 8), the full gradient becomes:

$$\frac{\partial L}{\partial u} = \sum_{k=1}^{m} \left( M_k \frac{\partial Q_{sc}}{\partial u} + \frac{\partial Q_{kc}}{\partial u} \right) \alpha_c, \quad \text{where} \quad M_k = -\frac{1}{\zeta}(Q_{ks}(Q_{ss}^{-1} - \upsilon\upsilon^\top) + y_k \upsilon^\top)$$

**Reference:** Equations (3), (6), (7), (8), (9), (10), Section 2.1  

**Purpose:**  
This closed-form gradient tells the attacker exactly which direction to move $x_c$ to most increase the validation error, while guaranteeing the SVM remains at its optimum. This is the central technical novelty of the paper.

---
### Step 5: Kernelise the Gradient

**What happens:**  
Equation (10) depends on $x_c$ only through the kernel gradient terms $\partial Q_{ic} / \partial u = y_i y_c \cdot \partial K(x_i, x_c) / \partial u$. The paper provides explicit closed-form expressions for these for three standard kernels:

| Kernel | $\partial K_{ic} / \partial u$ |
|--------|--------------------------------|
| Linear | $t \cdot x_i$ |
| Polynomial | $d(x_i \cdot x_c + R)^{d-1} \cdot t \cdot x_i$ |
| RBF | $K(x_i, x_c) \cdot \gamma t (x_i - x_c)$ |

For non-linear kernels, $x_c^{(p)}$ in the gradient is approximated by $x_c^{(p-1)}$ when the step size $t$ is small.

**Reference:** Section 2.2  

**Purpose:**  
This is the key advantage over prior work. Because the gradient is expressed purely in terms of kernel dot products (not inner products in the feature space $\phi(x)$), the entire attack is executable in the **original input space** $\mathbb{R}^d$ — even for non-linear kernels. Previous methods (e.g., Brückner & Scheffer, 2009; Kloft & Laskov, 2010) required working in the feature space, which a real attacker cannot access.

---
### Step 6: Update the Attack Point via Gradient Ascent

**What happens:**  
The attack point is updated by moving one small step in the direction of the gradient:

$$x_c^{(p)} = x_c^{(p-1)} + t \cdot u$$

where $u$ is the unit vector aligned with $\partial L / \partial u$ and $t$ is a fixed small step size (not a line search, because large steps break the incremental SVM assumption).

**Reference:** Algorithm 1 lines 6–7, Section 2.3  

**Purpose:**  
Iteratively moves the attack point toward a local maximum of the validation error surface. The fixed step size keeps the SVM sets $S$, $E$, $R$ (margin SVs, error SVs, reserve points) intact between iterations, which is a prerequisite for the adiabatic gradient derivation in Step 4.

---
### Step 7: Terminate and Return the Poisoning Point

**What happens:**  
The loop (Steps 3–6) repeats until the improvement in validation loss between two consecutive iterations falls below a threshold $\epsilon$:

$$L(x_c^{(p)}) - L(x_c^{(p-1)}) < \epsilon$$

For linear kernels (where the error surface is unbounded), an additional bounding box constraint on $x_c$ is imposed to prevent the attack point from drifting arbitrarily far from the training data. The final $x_c$ is returned as the poisoning point.

**Reference:** Algorithm 1 line 8, termination condition, Section 2.3  

**Purpose:**  
Despite the non-convexity of $L(x_c)$, the authors show experimentally (Fig. 1, Section 3.1) that the gradient ascent reliably converges to good local maxima that cause a significant increase in test error — from an initial 2–5% error up to 15–20% using a single injected point (Section 3.2, MNIST experiments).

---
## Final Summary Sentence

This paper solves the problem of how an adversary can craft a single malicious training point to maximally degrade an SVM classifier, and the authors claim their approach is superior to prior work because it is the **first poisoning method that operates entirely in the original input space for non-linear kernels** — made possible by kernelising the closed-form gradient derived from the SVM's KKT optimality conditions — whereas previous methods required feature-space access that is practically unavailable to a real attacker.